In [1]:
import os
for v in ['OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS','NUMEXPR_NUM_THREADS']: os.environ[v]='1'
import pandas as pd, numpy as np, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
THESIS = Path(r"Z:\Users\predicting recovery\AmanDeep\Thesis")
DATA_RAW = THESIS/"data_raw"; DATA_DERIVED = THESIS/"data_derived"
DATA_DERIVED.mkdir(parents=True, exist_ok=True)
DUCA_FILE = DATA_RAW/"Geleijn_duca_2025_data.xlsx"
print('reading', DUCA_FILE)

reading Z:\Users\predicting recovery\AmanDeep\Thesis\data_raw\Geleijn_duca_2025_data.xlsx


In [2]:
sheets=pd.read_excel(DUCA_FILE, sheet_name=None)
duca=sheets['duca_2016-2025']; overig=sheets['Data overig_Upper GI database ']
def pad(s):
    x=pd.to_numeric(s,errors='coerce'); return x.astype('Int64').astype(str).str.zfill(7).where(x.notna(),other=pd.NA)
duca['upn']=pad(duca['upn']); overig['upn']=pad(overig['upn'])
merged=duca.merge(overig,on='upn',how='outer',suffixes=('','_ovr'))
merged['in_duca']=merged['upn'].isin(set(duca['upn'].dropna())); merged['in_overig']=merged['upn'].isin(set(overig['upn'].dropna()))
print('merged', merged.shape)

merged (1201, 294)


In [3]:
SENT={'asascore':[9],'whopreneo':[9],'neoadj':[9],'neoadjtype':[9],'neoadjaf':[9],'heropn':[9],
      'lengte':[999],'gewicht':[999],'gewverl':[999],'packyears':[999],'tumlok1':[9,98],'tumhist':[9],
      'scorect':[99],'scorecn':[9],'conversie':[9],'anamok':[9],'anammed2':[9],'dietist':[9],
      'procok':[9],'resecok':[77],'analig':[9],'recontype':[9],'compper':[9],'curaok':[9],
      'status':[9],'reint':[9],'comphopn':[9],'icdg':[999],
      'commyo':[9],'comhartfaal':[9],'comaf':[9],'comcarritme':[9],'comlong':[9],'comperif':[9],'comcva':[9],
      'comdem':[9],'comparalys':[9],'combind':[9],'comgiulcus':[9],'comlever':[9],'compancr':[9],'comdiam':[9],
      'comnier':[9],'comhiv':[9],'commalig':[9],'leukemie':[9],'lymfoom':[9]}
for c in merged.columns:
    if c.startswith('clav'): SENT.setdefault(c,[9])
for col,bad in SENT.items():
    if col in merged.columns and pd.api.types.is_numeric_dtype(merged[col]):
        merged.loc[merged[col].isin(bad),col]=np.nan
NEG=[-95,-96,-97,-98,-99]
AUX={'packyears':(0,150),'bloedverlies':(0,10000),'duur_operatie':(1,900),'icdg':(0,365)}
for c,(lo,hi) in AUX.items():
    if c in merged.columns:
        s=pd.to_numeric(merged[c],errors='coerce'); s=s.where(~s.isin(NEG+[999,9999,99999]),np.nan)
        merged[c]=s.where((s>=lo)&(s<=hi),np.nan)
print('sentinels done')

sentinels done


In [4]:
merged['datok']=pd.to_datetime(merged['datok'],errors='coerce'); merged['gebdat']=pd.to_datetime(merged['gebdat'],errors='coerce')
merged['datovl']=pd.to_datetime(merged['datovl'],errors='coerce')
merged['age_at_surgery']=((merged['datok']-merged['gebdat']).dt.days/365.25).round(2)
merged['surgery_year']=merged['datok'].dt.year
merged['surgery_date']=merged['datok'].dt.strftime('%Y-%m-%d')
los0=pd.to_numeric(merged['opnameduur'],errors='coerce'); merged.loc[(los0<=0)&los0.notna(),'opnameduur']=np.nan
n0=len(merged)
merged=merged[~(merged['age_at_surgery'].notna()&((merged['age_at_surgery']<18)|(merged['age_at_surgery']>100)))].copy()
merged=merged.sort_values('datok').drop_duplicates(subset='upn',keep='first').copy()
print('Excl values present:'); print(merged['Excl'].astype(str).str.lower().str.strip().value_counts().head(12).to_string())
excl=merged['Excl'].astype(str).str.lower().str.strip()
exclude=excl.str.contains('salvage',na=False)|excl.str.contains('prophyl',na=False)|(merged['in_duca'].eq(False)&merged['in_overig'].eq(True))
cohort=merged[~exclude].copy()
has_e=cohort[['stadpt','stadpn']].notna().any(axis=1); has_g=cohort[['stadptmaag','stadpnmaag']].notna().any(axis=1)
cohort['cancer_subtype']=np.select([has_e&~has_g,has_g&~has_e,has_e&has_g],['oesophageal','gastric','junction'],default='unclassified')
print('subtype counts:'); print(cohort['cancer_subtype'].value_counts(dropna=False).to_string())
INCLUDE_JUNCTION=False  # GEJ/junction: confirm with Hidde whether to treat as oesophageal
cohort=cohort[cohort['cancer_subtype'].isin(['oesophageal']+(['junction'] if INCLUDE_JUNCTION else []))].copy()
print('oesophageal-only cohort', len(cohort))
print('\ncohort', len(cohort), 'from', n0)

Excl values present:
nan                             1122
1, salvage resectie               51
1, palliatief                     12
mogelijk, onbekende resectie       5
1 prophylactische resectie         2
subtype counts:
oesophageal     1022
gastric           62
unclassified      37
oesophageal-only cohort 1022

cohort 1022 from 1201


In [5]:
h=pd.to_numeric(cohort['lengte'],errors='coerce')/100; w=pd.to_numeric(cohort['gewicht'],errors='coerce')
cohort['bmi']=(w/(h**2)).round(2); cohort.loc[(cohort['bmi']<10)|(cohort['bmi']>70),'bmi']=np.nan
cohort['sex_male']=(pd.to_numeric(cohort['geslacht'],errors='coerce')==1).astype('Int64')
cohort['asascore']=pd.to_numeric(cohort['asascore'],errors='coerce')
cohort['comlong']=pd.to_numeric(cohort['comlong'],errors='coerce')
cohort['comdiam_ord']=pd.to_numeric(cohort['comdiam'],errors='coerce')
gv=pd.to_numeric(cohort['gewverl'],errors='coerce'); cohort['weight_loss_kg']=gv.where((gv>=0)&(gv<=100),np.nan)
cohort['tumlok1']=pd.to_numeric(cohort['tumlok1'],errors='coerce')
def grp(cols):
    pres=[c for c in cols if c in cohort.columns]; return (cohort[pres].apply(pd.to_numeric,errors='coerce')>=1).any(axis=1).astype('Int64')
cohort['group_cardiovascular']=grp(['commyo','comhartfaal','comaf','comcarritme'])
cohort['anamok_prior_surgery']=(pd.to_numeric(cohort['anamok'],errors='coerce')==1).astype('Int64')
cohort['immunosuppression']=(pd.to_numeric(cohort['anammed2'],errors='coerce')==1).astype('Int64')
cohort['dietitian_preop']=(pd.to_numeric(cohort['dietist'],errors='coerce')==1).astype('Int64')
# neoadjuvant
cohort['neoadj_received']=(pd.to_numeric(cohort['neoadj'],errors='coerce')==1).astype('Int64')
cohort['neoadj_chemoradiation']=(pd.to_numeric(cohort['neoadjtype'],errors='coerce')==2).astype('Int64')
cohort['neoadj_completed']=(pd.to_numeric(cohort['neoadjaf'],errors='coerce')==1).astype('Int64')
# clinical stage and histology
cohort['ct_stage']=pd.to_numeric(cohort['scorect'],errors='coerce')
cohort['cn_stage']=pd.to_numeric(cohort['scorecn'],errors='coerce')
cohort['histology']=pd.to_numeric(cohort['tumhist'],errors='coerce')
cohort['histology_adeno']=(cohort['histology']==1).astype('Int64')
cohort['histology_squamous']=(cohort['histology']==2).astype('Int64')
print('preop predictors derived')

preop predictors derived


In [6]:
rk=cohort['roken'].astype(str).str.strip(); rk=rk.where(~rk.isin(['nan','##USER_MISSING_99##','Onbekend','None']),np.nan)
cohort['roken_cat']=rk
cohort['smoking_current']=np.where(rk.isna(),np.nan,(rk=='Momenteel').astype(float))
cohort['smoking_ever']=np.where(rk.isna(),np.nan,rk.isin(['Momenteel','Gestopt']).astype(float))
py=pd.to_numeric(cohort['packyears'],errors='coerce')
chk=pd.DataFrame({'roken':rk,'has_py':py.notna()})
print(chk.groupby('roken')['has_py'].agg(['size','sum']).rename(columns={'sum':'with_packyears'}).to_string())
print('never smokers with pack years (expect ~0):', int(chk[chk['roken']=='Nooit']['has_py'].sum()))

               size  with_packyears
roken                              
Gestopt         376             167
Momenteel       141              72
Nooit gerookt   226               0
never smokers with pack years (expect ~0): 0


In [7]:
def b(col): return (pd.to_numeric(cohort[col],errors='coerce')>=1).astype(float) if col in cohort.columns else pd.Series(0.0,index=cohort.index)
dia=pd.to_numeric(cohort['comdiam'],errors='coerce')
cci=pd.Series(0.0,index=cohort.index)
for c in ['commyo','comhartfaal','comperif','comcva','comdem','comlong','combind','comgiulcus']: cci=cci+b(c)
cci=cci+np.where(b('compancr')>=1,3.0,np.where(b('comlever')>=1,1.0,0.0))
cci=cci+np.where(dia>=2,2.0,np.where(dia==1,1.0,0.0))
cci=cci+2*b('comparalys')+2*b('comnier')
cci=cci+np.where((b('commalig')+b('leukemie')+b('lymfoom'))>=1,2.0,0.0)
cci=cci+6*b('comhiv')
cohort['charlson_cci']=cci
print('CCI mean',round(cci.mean(),2),'median',cci.median()); print(cohort['charlson_cci'].value_counts().sort_index().to_string())

CCI mean 0.97 median 1.0
0.0    510
1.0    234
2.0    164
3.0     65
4.0     24
5.0     17
6.0      6
7.0      2


In [8]:
appr=pd.to_numeric(cohort['procok'],errors='coerce')
cohort['surgical_approach_mis']=np.where(appr.isna(),np.nan,(appr.isin([2,3,4])).astype(float))  # 1=open, 2-4=minimally invasive
cohort['resection_type']=pd.to_numeric(cohort['resecok'],errors='coerce')        # 0 transcerv,1 transhiatal,2 transthoracic,3 total gastr,4 partial gastr
cohort['resection_transthoracic']=(cohort['resection_type']==2).astype('Int64')
ana=pd.to_numeric(cohort['analig'],errors='coerce')
cohort['anastomosis_location']=ana                                               # 1 cervical,2 intrathoracic,3 intra-abdominal
cohort['anastomosis_cervical']=(ana==1).astype('Int64')
cohort['reconstruction_type']=pd.to_numeric(cohort['recontype'],errors='coerce')
cohort['op_duration_min']=pd.to_numeric(cohort['duur_operatie'],errors='coerce')
cohort['blood_loss_ml']=pd.to_numeric(cohort['bloedverlies'],errors='coerce')
conv=pd.to_numeric(cohort['conversie'],errors='coerce')                          # 0 no,1 early,2 late
cohort['conversion']=np.where(conv.isna(),np.nan,(conv.isin([1,2])).astype(float))
cohort['intraop_complication']=(pd.to_numeric(cohort['compper'],errors='coerce')==1).astype('Int64')
for c in ['surgical_approach_mis','resection_type','anastomosis_location','reconstruction_type','op_duration_min','blood_loss_ml','conversion','intraop_complication']:
    print(f"  {c:24s} missing {100*pd.to_numeric(cohort[c],errors='coerce').isna().mean():5.1f}%")

  surgical_approach_mis    missing   0.1%
  resection_type           missing   0.6%
  anastomosis_location     missing   0.8%
  reconstruction_type      missing   0.8%
  op_duration_min          missing  25.6%
  blood_loss_ml            missing  34.3%
  conversion               missing   8.2%
  intraop_complication     missing   0.0%


In [9]:
cohort['icu_days']=pd.to_numeric(cohort['icdg'],errors='coerce')
cohort['icu_readmission']=(pd.to_numeric(cohort['comphopn'],errors='coerce')==1).astype('Int64')
print('icu_days median', cohort['icu_days'].median(), 'missing', round(100*cohort['icu_days'].isna().mean(),1),'%')
print('icu_readmission prev', round(100*(cohort['icu_readmission']==1).mean(),1),'%')

icu_days median 0.0 missing 0.2 %
icu_readmission prev 8.6 %


In [10]:
clav=[c for c in cohort.columns if c.startswith('clav')]
cohort['cd_max']=cohort[clav].apply(pd.to_numeric,errors='coerce').max(axis=1,skipna=True)
cn=pd.to_numeric(cohort['compl'],errors='coerce')
cohort['any_complication']=cn.astype('Int64')
def cd_ge(t):
    r=(cohort['cd_max']>=t).astype(float)
    return r.where(cohort['cd_max'].notna(),other=pd.Series(np.where(cn==0,0.0,np.nan),index=cohort.index)).astype('Int64')
cohort['cd_ge_II']=cd_ge(2); cohort['cd_ge_IIIa']=cd_ge(3); cohort['cd_ge_IIIb']=cd_ge(4); cohort['cd_ge_IV']=cd_ge(5)
cohort['mortality_cd_V']=cd_ge(7)
cp=pd.to_numeric(cohort['comppulm'],errors='coerce')
cohort['pulmonary']=cp.where(cp.notna(),other=pd.Series(np.where(cn==0,0.0,np.nan),index=cohort.index)).astype('Int64')
cohort['readmission']=pd.to_numeric(cohort['heropn'],errors='coerce').astype('Int64')
cohort['reintervention']=(pd.to_numeric(cohort['reint'],errors='coerce')==1).astype('Int64')
los=pd.to_numeric(cohort['opnameduur'],errors='coerce'); cohort['los_days']=los
PROLONGED_CUTOFF_DAYS=7  # normal discharge day 6 to 7; prolonged = more than 7 days (6 June meeting)
thr=los.quantile(0.75)  # retained only for the Textbook Outcome length-of-stay component
cohort['los_prolonged']=pd.Series(np.where(los.notna(),(los>PROLONGED_CUTOFF_DAYS).astype(float),np.nan),index=cohort.index).astype('Int64')
cohort['los_extended']=pd.Series(np.where(los.notna(),(los>thr).astype(float),np.nan),index=cohort.index).astype('Int64')
# mortality: 30-day from status (direct), 90-day from death date
cohort['mortality_30d']=(pd.to_numeric(cohort['status'],errors='coerce')==1).astype('Int64')
dtd=(cohort['datovl']-cohort['datok']).dt.days
cohort['mortality_90d']=np.where(cohort['datovl'].notna()&(dtd>=0)&(dtd<=90),1,0)
cohort['mortality_90d']=cohort['mortality_90d'].astype('Int64')
# discharge home and Textbook Outcome composite
onts=pd.to_numeric(cohort['ontslag'],errors='coerce')
cohort['discharge_home']=np.where(onts.isna(),np.nan,(onts==1).astype(float))
to_parts={'no_major':(cohort['cd_ge_IIIb']==0),'no_reint':(cohort['reintervention']==0),
          'no_readm':(cohort['readmission']==0),'not_prolonged':(cohort['los_extended']==0),
          'home':(cohort['discharge_home']==1),'alive':(cohort['mortality_30d']==0)}
to_df=pd.DataFrame(to_parts); known=to_df.notna().all(axis=1)
cohort['textbook_outcome']=np.where(known,(to_df.all(axis=1)).astype(float),np.nan)
cohort['textbook_outcome']=cohort['textbook_outcome'].astype('Int64')
print('LOS p75',round(float(thr),1),'| prolonged cutoff days',PROLONGED_CUTOFF_DAYS)
_lp=pd.to_numeric(cohort['los_prolonged'],errors='coerce')
print('prolonged stay rate and n by surgery_year:')
print(pd.DataFrame({'rate':_lp.groupby(cohort['surgery_year']).mean().round(3),'n':_lp.groupby(cohort['surgery_year']).apply(lambda s:int(s.notna().sum()))}).to_string())
for o in ['any_complication','cd_ge_II','cd_ge_IIIa','cd_ge_IIIb','cd_ge_IV','mortality_cd_V','mortality_30d','mortality_90d','pulmonary','readmission','reintervention','los_prolonged','discharge_home','textbook_outcome']:
    s=pd.to_numeric(cohort[o],errors='coerce'); k=int(s.notna().sum())
    print(f"  {o:18s} events={int((s==1).sum()):4d} known={k:4d} prev={100*(s==1).sum()/k:.1f}%" if k else f"  {o}: none")

LOS p75 15.0 | prolonged cutoff days 7
prolonged stay rate and n by surgery_year:
               rate    n
surgery_year            
2016.0        0.817  115
2017.0        0.742  120
2018.0        0.721  111
2019.0         0.75  100
2020.0        0.742   89
2021.0        0.747   75
2022.0        0.663   92
2023.0        0.559  102
2024.0          0.6   95
2025.0        0.533  122
  any_complication   events= 588 known=1022 prev=57.5%
  cd_ge_II           events= 481 known=1017 prev=47.3%
  cd_ge_IIIa         events= 267 known=1017 prev=26.3%
  cd_ge_IIIb         events= 154 known=1017 prev=15.1%
  cd_ge_IV           events= 109 known=1017 prev=10.7%
  mortality_cd_V     events=  16 known=1017 prev=1.6%
  mortality_30d      events=  21 known=1022 prev=2.1%
  mortality_90d      events=  30 known=1022 prev=2.9%
  pulmonary          events= 246 known=1022 prev=24.1%
  readmission        events= 150 known=1008 prev=14.9%
  reintervention     events= 234 known=1022 prev=22.9%
  los_prolonged 

In [11]:
ID=['upn','cancer_subtype','surgery_year','surgery_date']
PREOP=['age_at_surgery','sex_male','bmi','asascore','comlong','comdiam_ord','group_cardiovascular',
       'charlson_cci','weight_loss_kg','smoking_current','smoking_ever','anamok_prior_surgery','immunosuppression',
       'dietitian_preop','neoadj_received','neoadj_chemoradiation','neoadj_completed','tumlok1','ct_stage','cn_stage',
       'histology','histology_adeno','histology_squamous']
OPER=['surgical_approach_mis','resection_type','resection_transthoracic','anastomosis_location','anastomosis_cervical',
      'reconstruction_type','op_duration_min','blood_loss_ml','conversion','intraop_complication']
DURING=['icu_days','icu_readmission']
OUTCOMES=['any_complication','cd_max','cd_ge_II','cd_ge_IIIa','cd_ge_IIIb','cd_ge_IV','mortality_cd_V','mortality_30d',
          'mortality_90d','pulmonary','readmission','reintervention','los_days','los_prolonged','los_extended','discharge_home','textbook_outcome']
keep=[c for c in ID+PREOP+OPER+DURING+OUTCOMES if c in cohort.columns]
clean=cohort[keep].copy()
clean.to_csv(DATA_DERIVED/"cohort_clean.csv", index=False)
def grp_of(c):
    return ('id' if c in ID else 'preoperative' if c in PREOP else 'operative' if c in OPER else 'during_stay' if c in DURING else 'outcome')
pd.DataFrame({'column':keep,'group':[grp_of(c) for c in keep]}).to_csv(DATA_DERIVED/"cohort_columns.csv", index=False)
print('saved cohort_clean.csv', clean.shape)
print(pd.Series([grp_of(c) for c in keep]).value_counts().to_string())

saved cohort_clean.csv (1022, 56)
preoperative    23
outcome         17
operative       10
id               4
during_stay      2
